In [2]:
#Set up Python env (pandas, numpy, matplotlib/seaborn).
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

In [3]:
#Load these CSVs into DataFrames: drivers.csv, races.csv, results.csv, constructors.csv, circuits.csv
df_drivers = pd.read_csv('../data/drivers.csv')
df_races = pd.read_csv('../data/races.csv')
df_results = pd.read_csv('../data/results.csv')
df_constructors = pd.read_csv('../data/constructors.csv')
df_circuits = pd.read_csv('../data/circuits.csv')

In [4]:
#Print .shape and .dtypes for each DataFrame
dfs = {
    "drivers": df_drivers,
    "races": df_races,
    "results": df_results,
    "constructors": df_constructors,
    "circuits": df_circuits
}

for name, df in dfs.items():
    print(f"===== {name} =====")
    print("shape:", df.shape)
    print(df.dtypes)
    print()

===== drivers =====
shape: (861, 9)
driverId       int64
driverRef        str
number           str
code             str
forename         str
surname          str
dob              str
nationality      str
url              str
dtype: object

===== races =====
shape: (1125, 18)
raceId         int64
year           int64
round          int64
circuitId      int64
name             str
date             str
time             str
url              str
fp1_date         str
fp1_time         str
fp2_date         str
fp2_time         str
fp3_date         str
fp3_time         str
quali_date       str
quali_time       str
sprint_date      str
sprint_time      str
dtype: object

===== results =====
shape: (26759, 18)
resultId             int64
raceId               int64
driverId             int64
constructorId        int64
number                 str
grid                 int64
position               str
positionText           str
positionOrder        int64
points             float64
laps                 i

In [5]:
# Identify missing/placeholder values (e.g. '\N') in results.csv (columns: milliseconds, fastestLapTime, fastestLapSpeed) and drivers.csv (dob, nationality)
results_cols = ['milliseconds', 'fastestLapTime', 'fastestLapSpeed']
df_results.replace('\\N', np.nan, inplace=True)
for c in results_cols:
    print(f'results missing {c} counts : {df_results[c].isnull().sum()}')

drivers_cols = ['dob', 'nationality']
df_drivers.replace('\\N', np.nan, inplace=True)
for c in drivers_cols:
    print(f'drivers missing {c} counts : {df_drivers[c].isnull().sum()}')

results missing milliseconds counts : 19079
results missing fastestLapTime counts : 18507
results missing fastestLapSpeed counts : 18507
drivers missing dob counts : 0
drivers missing nationality counts : 0


In [6]:
# Report count of missing values per column BEFORE and AFTER cleaning. Convert data types where needed (e.g. dob to datetime, milliseconds to numeric)
missing_before_drivers = df_drivers.isnull().sum()
missing_before_results = df_results.isnull().sum()

print("=== BEFORE cleaning ===")
print("=== drivers ===")
print(missing_before_drivers)
print("=== results ===")
print(missing_before_results)

df_drivers.replace('\\N', np.nan, inplace=True)
df_results.replace('\\N', np.nan, inplace=True)

df_drivers['dob'] = pd.to_datetime(df_drivers['dob'], errors='coerce')
df_results['milliseconds'] = pd.to_numeric(df_results['milliseconds'], errors='coerce')

print(df_drivers['dob'].dtype)       
print(df_results['milliseconds'].dtype) 

missing_after_drivers = df_drivers.isnull().sum()
missing_after_results = df_results.isnull().sum()
print("=== AFTER cleaning ===")
print("=== drivers ===")
print(missing_after_drivers)
print("=== results ===")
print(missing_after_results)

=== BEFORE cleaning ===
=== drivers ===
driverId         0
driverRef        0
number         802
code           757
forename         0
surname          0
dob              0
nationality      0
url              0
dtype: int64
=== results ===
resultId               0
raceId                 0
driverId               0
constructorId          0
number                 6
grid                   0
position           10953
positionText           0
positionOrder          0
points                 0
laps                   0
time               19079
milliseconds       19079
fastestLap         18507
rank               18249
fastestLapTime     18507
fastestLapSpeed    18507
statusId               0
dtype: int64
datetime64[us]
float64
=== AFTER cleaning ===
=== drivers ===
driverId         0
driverRef        0
number         802
code           757
forename         0
surname          0
dob              0
nationality      0
url              0
dtype: int64
=== results ===
resultId               0
raceId    

In [7]:
# Merge results + races + drivers + constructors into one DataFrame with at least: raceId, year, round, driverName (full name), constructorName, grid, position, points, statusId
df_drivers['full_name'] = df_drivers[['forename', 'surname']].agg(' '.join, axis=1)
drivers_subset = df_drivers[['driverId', 'full_name']]
races_subset = df_races[['raceId', 'year', 'round']]
results_subset = df_results[['raceId', 'driverId', 'constructorId', 'grid', 'position', 'points', 'statusId']]
constructors_subset = df_constructors[['constructorId', 'name']]

merged = results_subset.merge(drivers_subset, on = 'driverId', how = 'left') \
                        .merge(races_subset, on = 'raceId', how = 'left') \
                        .merge(constructors_subset, on = 'constructorId', how = 'left')

print(merged.shape)
print(merged.columns)
merged.head()

(26759, 11)
Index(['raceId', 'driverId', 'constructorId', 'grid', 'position', 'points',
       'statusId', 'full_name', 'year', 'round', 'name'],
      dtype='str')


,raceId,driverId,constructorId,grid,position,points,statusId,full_name,year,round,name
0,18,1,1,1,1,10.0,1,Lewis Hamilton,2008,1,McLaren
1,18,2,2,5,2,8.0,1,Nick Heidfeld,2008,1,BMW Sauber
2,18,3,3,7,3,6.0,1,Nico Rosberg,2008,1,Williams
3,18,4,4,11,4,5.0,1,Fernando Alonso,2008,1,Renault
4,18,5,1,3,5,4.0,1,Heikki Kovalainen,2008,1,McLaren


In [8]:
#Export result as merged_f1.csv.
merged.to_csv('../data/merged_f1.csv', index=False)

In [9]:
# Using groupby/aggregation, compute total career points per driver (all seasons, 1950-2020)
filtered = merged[(merged['year'] >= 1950) & (merged['year'] <= 2020)]
career = filtered.groupby('full_name')['points'].sum()
print(career)

full_name
Adolf Brudes           0.0
Adolfo Cruz            0.0
Adrian Sutil         124.0
Adrián Campos          0.0
Aguri Suzuki           8.0
                     ...  
Zsolt Baumgartner      1.0
Élie Bayol             2.0
Éric Bernard          10.0
Érik Comas             7.0
Óscar González         0.0
Name: points, Length: 850, dtype: float64
